# Pipeline de Dados — Acidentes de Trânsito 2025
## João Monlevade (MG) — REDS/Sejusp-MG

In [1]:
import pandas as pd

## 1. Carga dos arquivos brutos

In [2]:
condicoes = pd.read_csv(
    "2025/Condições - Acidente de Trânsito - MG - Jan a Dez 2025.csv",
    sep=None, engine="python", encoding="utf-8-sig"
)
acidentes = pd.read_csv(
    "2025/Registros de Acidente de Trânsito - MG - Jan a Dez 2025 (2).csv",
    sep=None, engine="python", encoding="utf-8-sig"
)
veiculos = pd.read_csv(
    "2025/Veículos envolvidos em Acidente de Trânsito - MG - Jan a Dez 2025.csv",
    sep=None, engine="python", encoding="utf-8-sig"
)
vias = pd.read_csv(
    "2025/Via - Acidente de Trânsito - MG - Jan a Dez 2025.csv",
    sep=None, engine="python", encoding="utf-8-sig"
)
vitimas = pd.read_csv(
    "2025/Vítimas de Acidente de Trânsito - MG - Jan a Dez 2025.csv",
    sep=None, engine="python", encoding="utf-8-sig"
)
print(f'acidentes:  {acidentes.shape}')
print(f'condicoes:  {condicoes.shape}')
print(f'vias:       {vias.shape}')
print(f'vitimas:    {vitimas.shape}')
print(f'veiculos:   {veiculos.shape}')

acidentes:  (312700, 25)
condicoes:  (83637, 8)
vias:       (77442, 11)
vitimas:    (90295, 23)
veiculos:   (551161, 5)


## 2. Filtro — João Monlevade

Filtramos antes do merge para trabalhar só com as ~1.012 ocorrências do município.

In [ ]:
import unicodedata

def normalizar_str(s):
    """Remove acentos e converte para maiúsculas — robusto a variações do REDS."""
    return unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode().upper()

acidentes_jm = acidentes[
    acidentes["nome_municipio"].apply(normalizar_str).str.contains("JOAO MONLEVADE", na=False)
].copy().reset_index(drop=True)

print(f'Acidentes em João Monlevade: {len(acidentes_jm)}')

## 3. Agregação das tabelas N:1 antes do merge

**Por que este passo é necessário?**

Três tabelas têm mais de uma linha por ocorrência:

| Tabela   | Motivo                                          |
|----------|-------------------------------------------------|
| condicoes | Até 4 linhas por acidente (condições por faixa) |
| vitimas   | 1 linha por pessoa envolvida                    |
| veiculos  | 1 linha por veículo envolvido                   |

Sem agregar antes, o merge multiplica as linhas:
acidente com 2 veículos → aparece 2 vezes no resultado final.
O dataset anterior tinha 602.445 linhas por esse motivo.

A solução é agregar cada tabela para **1 linha por ocorrência** antes de juntar.

### 3.1 Condições → 1 linha por ocorrência

Quando um acidente tem múltiplas linhas de condição (diferentes faixas da pista),
mantemos a **pior** condição registrada — relevante para análise de risco.
Ordem de prioridade: RUIM (0) > INEXISTENTE (1) > REGULAR (2) > BOA (3) > NAO INFORMADO (4)

> **Nota:** Esta é a tabela *Condições*, que registra estado da sinalização por faixa
> e pode ter até 4 linhas por acidente. Não confundir com a tabela *Via* (tipo de via,
> pavimento), que já é 1:1 e não precisa de agregação.

In [ ]:
# Prioridade: menor número = mantido quando há duplicata.
# Para análise de risco mantemos a PIOR condição registrada.
prioridade_sinalizacao = {
    "RUIM": 0, "INEXISTENTE": 1, "REGULAR": 2, "BOA": 3, "NAO INFORMADO": 4
}

condicoes["_prioridade"] = (
    condicoes["condicao_sinalizacao_descricao"]
    .str.upper().str.strip()
    .map(prioridade_sinalizacao)
    .fillna(5)
)

condicoes_agg = (
    condicoes
    .sort_values("_prioridade")
    .drop_duplicates(subset="numero_ocorrencia_associado", keep="first")
    .drop(columns=["_prioridade", "verificacao", "ano_fato", "mes_numerico_fato"])
    .reset_index(drop=True)
)

print(f'condicoes antes: {len(condicoes)} linhas')
print(f'condicoes_agg:   {len(condicoes_agg)} linhas  (1 por ocorrência)')

### 3.2 Vias — remover colunas redundantes

A tabela *Via* (tipo de via, pavimento, iluminação) é naturalmente **1:1** com acidentes —
verificado: 77.442 linhas para 77.442 IDs únicos em 2025.
Apenas removemos colunas já presentes em acidentes (ano, mês, verificacao).

In [5]:
vias_clean = vias.drop(columns=["verificacao", "ano_fato", "mes_numerico_fato"])

print(f'vias_clean: {vias_clean.shape}  (1 por ocorrência)')

vias_clean: (77442, 8)  (1 por ocorrência)


### 3.3 Vítimas → 1 linha por ocorrência

Colapsamos para 1 linha por acidente preservando:
- qtd_vitimas: total de envolvidos
- qtd_feridos: quantos tiveram LEVE ou GRAVE
- qtd_fatais: quantos morreram
- severidade_maxima: pior caso (FATAL > GRAVE > LEVE > ILESO)
- tipos_envolvimento: papéis presentes (ex: CONDUTOR|PEDESTRE)

In [ ]:
severidade_map = {
    "FATAL": 4, "ÓBITO": 4,
    "GRAVES": 3, "GRAVE": 3,
    "LEVES": 2, "LEVE": 2,
    "ILESO": 1, "SEM LESAO": 1, "SEM LESÃO": 1,
    "IGNORADO": 0, "NAO INFORMADO": 0
}

# Fix #6: padronizar strings ANTES do groupby para evitar duplicatas por espaço/caixa
vitimas["condicao_fisica_descricao"] = vitimas["condicao_fisica_descricao"].str.strip().str.upper()
vitimas["envolvimento_descricao"]    = vitimas["envolvimento_descricao"].str.strip().str.upper()

vitimas["_sev_num"] = (
    vitimas["condicao_fisica_descricao"]
    .map(severidade_map)
    .fillna(0)
)

vitimas_agg = vitimas.groupby("numero_ocorrencia_associado").agg(
    qtd_vitimas        = ("qtd_envolvido", "sum"),
    qtd_feridos        = ("condicao_fisica_descricao",
                          lambda x: x.isin(["LEVES","LEVE","GRAVES","GRAVE"]).sum()),
    qtd_fatais         = ("condicao_fisica_descricao",
                          lambda x: x.isin(["FATAL","ÓBITO"]).sum()),
    severidade_maxima_n = ("_sev_num", "max"),
    tipos_envolvimento = ("envolvimento_descricao",
                          lambda x: "|".join(sorted(x.dropna().unique())))
).reset_index()

sev_label = {4: "FATAL", 3: "GRAVE", 2: "LEVE", 1: "ILESO", 0: "NAO INFORMADO"}
vitimas_agg["severidade_maxima"] = vitimas_agg["severidade_maxima_n"].map(sev_label)
vitimas_agg = vitimas_agg.drop(columns="severidade_maxima_n")

print(f'vitimas antes: {len(vitimas)} linhas')
print(f'vitimas_agg:   {len(vitimas_agg)} linhas  (1 por ocorrência)')
print(vitimas_agg["severidade_maxima"].value_counts())

### 3.4 Veículos → 1 linha por ocorrência

Colapsamos para 1 linha por acidente preservando:
- qtd_veiculos: total de veículos
- tipos_veiculos: lista dos tipos (ex: AUTOMOVEL|MOTOCICLETA)
- flags tem_moto, tem_bicicleta, tem_caminhao, tem_onibus

In [ ]:
# Fix #6: padronizar strings ANTES do groupby para evitar duplicatas por espaço/caixa
# Fix #5: qtd_veiculo é sempre 1 (1 linha = 1 veículo) → count é semanticamente correto
veiculos["tipo_veiculo_descricao_longa"] = veiculos["tipo_veiculo_descricao_longa"].str.strip().str.upper()

veiculos_agg = veiculos.groupby("numero_ocorrencia_associado").agg(
    qtd_veiculos   = ("qtd_veiculo", "count"),  # cada linha = 1 veículo
    tipos_veiculos = ("tipo_veiculo_descricao_longa",
                      lambda x: "|".join(sorted(x.dropna().unique())))
).reset_index()

veiculos_agg["tem_moto"]      = veiculos_agg["tipos_veiculos"].str.contains("MOTOCICLETA", na=False)
veiculos_agg["tem_bicicleta"] = veiculos_agg["tipos_veiculos"].str.contains("BICICLETA", na=False)
veiculos_agg["tem_caminhao"]  = veiculos_agg["tipos_veiculos"].str.contains("CAMINHAO|CAMINHÃO", na=False)
veiculos_agg["tem_onibus"]    = veiculos_agg["tipos_veiculos"].str.contains("ONIBUS|ÔNIBUS", na=False)

print(f'veiculos antes: {len(veiculos)} linhas')
print(f'veiculos_agg:   {len(veiculos_agg)} linhas  (1 por ocorrência)')

## 4. Merge final

Todas as tabelas agora são 1:1 com acidentes.
Resultado esperado: exatamente 1.012 linhas, sem duplicatas.

In [ ]:
df_final = acidentes_jm.copy()

df_final = df_final.merge(condicoes_agg, on="numero_ocorrencia_associado", how="left")
df_final = df_final.merge(vias_clean,    on="numero_ocorrencia_associado", how="left")
df_final = df_final.merge(vitimas_agg,   on="numero_ocorrencia_associado", how="left")
df_final = df_final.merge(veiculos_agg,  on="numero_ocorrencia_associado", how="left")

# Fix #4: antes do fillna, proteger acidentes COM VÍTIMA que não cruzaram
# com vitimas_agg (dado ausente). Eles devem virar 'NAO INFORMADO', não 'SEM VITIMA'.
mask_com_vitima_sem_sev = (
    df_final["natureza_descricao"].str.upper().str.contains("COM VITIMA", na=False)
    & df_final["severidade_maxima"].isna()
)
df_final.loc[mask_com_vitima_sem_sev, "severidade_maxima"] = "NAO INFORMADO"
n_corr = mask_com_vitima_sem_sev.sum()
print(f'Acidentes COM VITIMA sem cruzamento corrigidos para NAO INFORMADO: {n_corr}')

# Agora sim: os NaN restantes são acidentes genuinamente sem vítima
df_final["severidade_maxima"] = df_final["severidade_maxima"].fillna("SEM VITIMA")
df_final["qtd_vitimas"]  = df_final["qtd_vitimas"].fillna(0).astype(int)
df_final["qtd_feridos"]  = df_final["qtd_feridos"].fillna(0).astype(int)
df_final["qtd_fatais"]   = df_final["qtd_fatais"].fillna(0).astype(int)
df_final["qtd_veiculos"] = df_final["qtd_veiculos"].fillna(0).astype(int)

print("Dimensões da base final:", df_final.shape)
print(f'Esperado: 1.012 linhas  |  Obtido: {len(df_final)} linhas')
print(f'Duplicatas na chave: {df_final["numero_ocorrencia_associado"].duplicated().sum()}')
display(df_final.head())

## 5. Limpeza dos dados

O diagnóstico executado sobre o dataset revelou os seguintes problemas reais:

| # | Problema encontrado | Ação |
|---|--------------------|----- |
| 1 | **Bairro 'INVALIDO'** — 83 registros com bairro literalmente cadastrado como 'INVALIDO' no REDS | Substituir por NaN |
| 2 | **Coordenadas com vírgula decimal** — `latitude_sirgas2000` e `longitude_sirgas2000` chegam como string com vírgula (`-19,88`) | Converter para float |
| 3 | **Strings sujas** — espaços extras, mistura de maiúsculas/minúsculas nas colunas de texto | Strip + upper |
| 4 | **Acidente COM VÍTIMA sem cruzamento** — quando o acidente tem natureza 'COM VÍTIMA' mas não há linha correspondente em vitimas_agg | Corrigir para 'NAO INFORMADO' antes do fillna |
| 5 | **Nulos em condições/vias/vítimas** — ocorrem porque **nem todo acidente gera registro** nessas tabelas no REDS, independentemente de ter vítima. Em JM, apenas 244 dos 1.012 acidentes têm dado de condições. | Manter — documentar por análise |
| 6 | **Nulos reais** — 35 sem coordenada (3,5%) e 36 sem logradouro (3,6%) — dado ausente na fonte | Manter no dataset, excluir só em análise espacial |

> Não há linhas duplicadas, datas fora de 2025 nem coordenadas fora do bounding box de João Monlevade.

> **Nota sobre coordenadas:** Os campos `latitude_sirgas2000` / `longitude_sirgas2000` usam o
> datum SIRGAS2000 (EPSG:4674). Para a escala municipal utilizada (1:10.000), a diferença
> em relação ao WGS84 (EPSG:4326) é inferior a 2 metros e pode ser desconsiderada.

In [ ]:
import numpy as np

df = df_final.copy()

# ── 5.1 Bairro 'INVALIDO' → NaN ──────────────────────────────────────────────
invalidos_antes = (df["nome_bairro"] == "INVALIDO").sum()
df["nome_bairro"] = df["nome_bairro"].replace("INVALIDO", np.nan)
print(f"5.1 Bairros 'INVALIDO' substituídos por NaN: {invalidos_antes}")

# ── 5.2 Coordenadas: vírgula → ponto, cast para float ────────────────────────
# SIRGAS2000 (EPSG:4674) — diferença < 2m em relação ao WGS84 na escala municipal.
for col in ["latitude_sirgas2000", "longitude_sirgas2000"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace("nan", np.nan)
        .astype(float)
    )
sem_coord = df[["latitude_sirgas2000","longitude_sirgas2000"]].isna().any(axis=1).sum()
print(f"5.2 Coordenadas convertidas para float. Sem coordenada: {sem_coord} registros")

# ── 5.3 Padronização de strings (strip + upper) ───────────────────────────────
cols_texto = [
    "natureza_descricao", "motivo_presumido_descricao",
    "tipo_logradouro_descricao", "logradouro_nome", "nome_bairro",
    "Grupo Local Imediato", "local_imediato_longa", "Dia da Semana",
    "luminosidade_descricao", "condicao_sinalizacao_descricao",
    "tipo_via_descricao", "tipo_pavimento_descricao",
]
for col in cols_texto:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
        )
print("5.3 Strings padronizadas (strip + upper) em", len(cols_texto), "colunas")

# ── 5.4 Verificação final de consistência severidade ─────────────────────────
# A correção principal (COM VÍTIMA sem cruzamento → NAO INFORMADO) foi feita
# na seção 4 (merge), antes do fillna. Aqui apenas confirmamos.
mask_inconsistente = (
    df["natureza_descricao"].str.contains("COM VITIMA", na=False) &
    (df["severidade_maxima"] == "SEM VITIMA")
)
print(f"5.4 Inconsistências COM VÍTIMA / SEM VITIMA remanescentes: {mask_inconsistente.sum()} (esperado: 0)")

# ── 5.5 Tipagem de data ───────────────────────────────────────────────────────
df["data_fato"] = pd.to_datetime(df["data_fato"], errors="coerce")
print(f"5.5 Datas convertidas. Nulas: {df['data_fato'].isna().sum()}")

# ── Resumo final ──────────────────────────────────────────────────────────────
print(f"\nShape após limpeza: {df.shape}")
print(f"Duplicatas na chave: {df['numero_ocorrencia_associado'].duplicated().sum()}")

## 6. Verificação dos nulos pós-limpeza

Os nulos têm três origens distintas:

| Origem | Colunas afetadas | Causa |
|--------|-----------------|-------|
| **Sem registro no REDS** | condições, vias, vítimas (~75%) | Nem todo acidente gera entrada nessas tabelas (ex.: sem vítima ou BO incompleto) |
| **Dado ausente na fonte** | coordenadas (3,5%), logradouro (3,6%), bairro (8,2%) | Campo não preenchido pelo registrador |
| **Corrigido intencionalmente** | `nome_bairro` | Valor 'INVALIDO' substituído por NaN (seção 5.1) |

> Para análise espacial (mapas, KDE) usar apenas os registros com coordenada válida.

In [ ]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
nulos_pct = (nulos / len(df) * 100).round(1)

# Categorizar origem dos nulos
origem_nulos = {
    "tipos_envolvimento":           "sem registro no REDS",
    "SULCOS HORIZONTAIS":           "sem registro no REDS",
    "ONDULADA":                     "sem registro no REDS",
    "MATERIAL NA PISTA":            "sem registro no REDS",
    "BOA":                          "sem registro no REDS",
    "OUTRAS (NO HISTORICO)":        "sem registro no REDS",
    "RACHADURA":                    "sem registro no REDS",
    "BURACOS":                      "sem registro no REDS",
    "tipo_via_descricao":            "sem registro no REDS",
    "condicao_sinalizacao_descricao":"sem registro no REDS",
    "tipo_pavimento_descricao":      "sem registro no REDS",
    "luminosidade_descricao":        "sem registro no REDS",
    "nome_bairro":                   "corrigido (INVALIDO→NaN) + ausente na fonte",
    "logradouro_nome":               "ausente na fonte",
    "latitude_sirgas2000":           "ausente na fonte",
    "longitude_sirgas2000":          "ausente na fonte",
    "motivo_presumido_descricao":     "ausente na fonte",
    "tipos_veiculos":                "ausente na fonte",
    "tem_moto":                      "ausente na fonte",
    "tem_bicicleta":                 "ausente na fonte",
    "tem_caminhao":                  "ausente na fonte",
    "tem_onibus":                    "ausente na fonte",
}

tabela_nulos = pd.DataFrame({"nulos": nulos, "pct (%)": nulos_pct})
tabela_nulos["origem"] = tabela_nulos.index.map(origem_nulos).fillna("verificar")

print("Colunas com nulos:")
print(tabela_nulos.to_string())

print("\nDistribuição de severidade (pós-limpeza):")
print(df["severidade_maxima"].value_counts())

print("\nDistribuição por tipo de logradouro:")
print(df["tipo_logradouro_descricao"].value_counts())

print("\nTop 10 bairros (sem INVALIDO):")
print(df["nome_bairro"].value_counts().head(10))

## 7. Engenharia de Atributos

Criamos colunas categóricas derivadas das colunas brutas. Essas colunas são
essenciais para a **Etapa 4 (mineração de regras de associação)**, que exige
atributos discretos e padronizados.

| Coluna criada | Derivada de | Categorias |
|---|---|---|
| `faixa_horaria` | `horario_fato` | Madrugada / Manhã / Tarde / Noite |
| `dia_semana_cat` | `Dia da Semana` | DIA_UTIL / FIM_DE_SEMANA |
| `mes_cat` | `mes_numerico_fato` | Agrupamento por trimestre |
| `severidade_cat` | `severidade_maxima` | SEM_VITIMA / LEVE / GRAVE_FATAL |
| `logradouro_cat` | `tipo_logradouro_descricao` | AVENIDA / RUA / RODOVIA / OUTRO |
| `motivo_cat` | `motivo_presumido_descricao` | 6 grupos de causa |
| `qtd_veiculos_cat` | `qtd_veiculos` | 1 / 2 / 3_OU_MAIS |
| `tipo_veiculo_cat` | `tem_moto`, `tem_caminhao`… | MOTO / CAMINHAO / ONIBUS / BICICLETA / CARRO |
| `tem_pedestre` | `tipos_envolvimento` | Sim / Não |
| `periodo_dia` | `luminosidade_descricao` | DIA / NOITE / CREPUSCULO |
| `pista_cat` | `tipo_via_descricao` | SIMPLES / DUPLA / MULTIPLA |
| `pavimento_cat` | `tipo_pavimento_descricao` | ASFALTO / OUTRO |


In [ ]:
# ── 7.1 Faixa horária ────────────────────────────────────────────────────────
hora = pd.to_datetime(df['horario_fato'], format='%H:%M:%S', errors='coerce').dt.hour

def classif_hora(h):
    if pd.isna(h): return np.nan
    if 0  <= h < 6:  return 'MADRUGADA'
    if 6  <= h < 12: return 'MANHA'
    if 12 <= h < 18: return 'TARDE'
    return 'NOITE'

df['faixa_horaria'] = hora.apply(classif_hora)
print('faixa_horaria:', df['faixa_horaria'].value_counts().to_dict())

# ── 7.2 Dia útil vs fim de semana ────────────────────────────────────────────
fim_de_semana = {'SÁBADO', 'DOMINGO'}
df['dia_semana_cat'] = df['Dia da Semana'].apply(
    lambda x: 'FIM_DE_SEMANA' if str(x).upper() in fim_de_semana else 'DIA_UTIL'
)
print('dia_semana_cat:', df['dia_semana_cat'].value_counts().to_dict())

# ── 7.3 Dia da semana (nome curto para análise e gráficos) ───────────────────
mapa_dia = {
    'SEGUNDA-FEIRA': 'SEGUNDA', 'TERÇA-FEIRA': 'TERCA', 'QUARTA-FEIRA': 'QUARTA',
    'QUINTA-FEIRA': 'QUINTA', 'SEXTA-FEIRA': 'SEXTA', 'SÁBADO': 'SABADO', 'DOMINGO': 'DOMINGO'
}
df['dia_semana_nome'] = df['Dia da Semana'].map(mapa_dia)
print('\nDia com mais acidentes:')
print(df['dia_semana_nome'].value_counts())

# ── 7.4 Trimestre ────────────────────────────────────────────────────────────
def trimestre(m):
    if pd.isna(m): return np.nan
    m = int(m)
    if m <= 3: return 'T1_JAN_MAR'
    if m <= 6: return 'T2_ABR_JUN'
    if m <= 9: return 'T3_JUL_SET'
    return 'T4_OUT_DEZ'

df['mes_cat'] = df['mes_numerico_fato'].apply(trimestre)
print('\nmes_cat:', df['mes_cat'].value_counts().to_dict())

# ── 7.5 Severidade simplificada (3 classes para mineração) ───────────────────
def sev_simples(s):
    if s in ('FATAL', 'GRAVE'): return 'GRAVE_FATAL'
    if s == 'LEVE':             return 'LEVE'
    if s == 'SEM VITIMA':       return 'SEM_VITIMA'
    return 'NAO_INFORMADO'

df['severidade_cat'] = df['severidade_maxima'].apply(sev_simples)
print('severidade_cat:', df['severidade_cat'].value_counts().to_dict())

# ── 7.6 Tipo de logradouro simplificado ──────────────────────────────────────
def logr_cat(t):
    if pd.isna(t) or str(t).upper() == 'NAN': return np.nan
    t = str(t).upper()
    if t == 'AVENIDA': return 'AVENIDA'
    if t == 'RUA':     return 'RUA'
    if t == 'RODOVIA': return 'RODOVIA'
    return 'OUTRO'

df['logradouro_cat'] = df['tipo_logradouro_descricao'].apply(logr_cat)
print('logradouro_cat:', df['logradouro_cat'].value_counts().to_dict())

# ── 7.7 Motivo agrupado em 7 categorias ──────────────────────────────────────
mapa_motivo = {
    'FALTA DE ATENCAO': 'ATENCAO', 'NAO MANTER DISTANCIA DE SEGURANCA': 'ATENCAO', 'DORMIR AO VOLANTE': 'ATENCAO',
    'MA VISIBILIDADE': 'VISIBILIDADE', 'AQUAPLANAGEM': 'VISIBILIDADE', 'DERRAPAGEM': 'VISIBILIDADE',
    'DESOBEDECER PARADA OBRIGATORIA': 'SINAL_VIA', 'AVANCO DE SINAL': 'SINAL_VIA',
    'CONTRAMAO DE DIRECAO': 'SINAL_VIA', 'ULTRAPASSAGEM FORCADA': 'SINAL_VIA',
    'ULTRAPASSAGEM PROIBIDA': 'SINAL_VIA', 'RETORNO/CONVERSAO EM LOCAL PROIBIDO': 'SINAL_VIA',
    'NAO DAR PREFERENCIA': 'SINAL_VIA',
    'VELOCIDADE INCOMPATIVEL': 'VELOCIDADE',
    'DIRIGIR EMBRIAGADO/ALCOOLIZADO': 'ALCOOL_DROGAS',
    'DEFEITO NO VEICULO': 'VEICULO_VIA', 'CARGA MAL ACONDICIONADA': 'VEICULO_VIA',
    'DEFEITO NA VIA': 'VEICULO_VIA', 'DEFEITO NA SINALIZACAO': 'VEICULO_VIA',
    'ANIMAL NA PISTA': 'VEICULO_VIA', 'LINHA COM CEROL (PAPAGAIO / PIPA)': 'VEICULO_VIA',
    'CONDUTA INADEQUADA DO PEDESTRE': 'PEDESTRE',
    'OUTROS - CAUSAS RELACIONADAS A TRANSITO': 'OUTROS',
}
df['motivo_cat'] = df['motivo_presumido_descricao'].map(mapa_motivo).fillna('OUTROS')
print('motivo_cat:', df['motivo_cat'].value_counts().to_dict())

# ── 7.8 Quantidade de veículos categorizada ───────────────────────────────────
def qtd_cat(n):
    if n == 1: return '1_VEICULO'
    if n == 2: return '2_VEICULOS'
    if n >= 3: return '3_OU_MAIS'
    return 'NAO_INFORMADO'

df['qtd_veiculos_cat'] = df['qtd_veiculos'].apply(qtd_cat)
print('qtd_veiculos_cat:', df['qtd_veiculos_cat'].value_counts().to_dict())

# ── 7.9 Tipo de veículo predominante ─────────────────────────────────────────
def tipo_veiculo_cat(row):
    if row.get('tem_moto')      == True: return 'MOTO'
    if row.get('tem_caminhao')  == True: return 'CAMINHAO'
    if row.get('tem_onibus')    == True: return 'ONIBUS'
    if row.get('tem_bicicleta') == True: return 'BICICLETA'
    return 'CARRO'

df['tipo_veiculo_cat'] = df.apply(tipo_veiculo_cat, axis=1)
print('tipo_veiculo_cat:', df['tipo_veiculo_cat'].value_counts().to_dict())

# ── 7.10 Tem pedestre envolvido ───────────────────────────────────────────────
df['tem_pedestre'] = df['tipos_envolvimento'].fillna('').str.contains('PEDESTRE').map(
    {True: 'SIM', False: 'NAO'}
)
print('tem_pedestre:', df['tem_pedestre'].value_counts().to_dict())

# ── 7.11 Período de iluminação — só MANHA/TARDE/NOITE (244 com dado) ──────────
# AMANHECER → MANHA  |  ENTARDECER → TARDE  (sem categoria crepúsculo)
def periodo_dia(lum):
    if pd.isna(lum) or str(lum).upper() in ('NAN', 'NAO INFORMADO'): return np.nan
    lum = str(lum).upper()
    if 'DIA' in lum or 'AMANHECER' in lum: return 'MANHA'
    if 'ENTARDECER' in lum:                return 'TARDE'
    return 'NOITE'

df['periodo_dia'] = df['luminosidade_descricao'].apply(periodo_dia)
print('periodo_dia:', df['periodo_dia'].value_counts().to_dict())

# ── 7.12 Tipo de pista (244 com dado) ────────────────────────────────────────
def pista_cat(t):
    if pd.isna(t) or str(t).upper() in ('NAN', 'NAO INFORMADO'): return np.nan
    t = str(t).upper()
    if 'SIMPLES'  in t: return 'SIMPLES'
    if 'DUPLA'    in t: return 'DUPLA'
    if 'MULTIPLA' in t: return 'MULTIPLA'
    return 'OUTRO'

df['pista_cat'] = df['tipo_via_descricao'].apply(pista_cat)
print('pista_cat:', df['pista_cat'].value_counts().to_dict())

# ── 7.13 Pavimento (244 com dado) ────────────────────────────────────────────
def pavimento_cat(t):
    if pd.isna(t) or str(t).upper() in ('NAN', 'NAO INFORMADO', 'OUTROS (NO HISTORICO)'): return np.nan
    if 'ASFALTO' in str(t).upper(): return 'ASFALTO'
    return 'OUTRO'

df['pavimento_cat'] = df['tipo_pavimento_descricao'].apply(pavimento_cat)
print('pavimento_cat:', df['pavimento_cat'].value_counts().to_dict())

cols_eng = ['faixa_horaria','dia_semana_cat','dia_semana_nome','mes_cat','severidade_cat',
            'logradouro_cat','motivo_cat','qtd_veiculos_cat','tipo_veiculo_cat',
            'tem_pedestre','periodo_dia','pista_cat','pavimento_cat']
print(f'\nShape após engenharia: {df.shape}  | colunas adicionadas: {len(cols_eng)}')


## 8. Análise Espacial — KDE (Kernel Density Estimation)

O KDE estima uma **superfície contínua de densidade** a partir dos pontos de acidente.
Diferente do heatmap do Folium (visual), este KDE usa `scipy.stats.gaussian_kde` com
banda otimizada (regra de Scott), gerando uma grade regular sobre João Monlevade.

- Cada célula da grade recebe a densidade estimada de acidentes
- Acidentes fatais/graves recebem peso maior (ponderado por severidade)
- O resultado é exportado como mapa interativo e como CSV de hotspots por rua


In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
os.makedirs('outputs', exist_ok=True)

# Base: acidentes com coordenada
df_geo = df.dropna(subset=['latitude_sirgas2000','longitude_sirgas2000']).copy()
centro_lat = df_geo['latitude_sirgas2000'].median()
centro_lon = df_geo['longitude_sirgas2000'].median()

# Pesos por severidade (fatais pesam mais)
peso_map = {'FATAL': 5, 'GRAVE': 4, 'LEVE': 3, 'NAO INFORMADO': 2, 'SEM VITIMA': 1}
pesos = df_geo['severidade_maxima'].map(peso_map).fillna(1).values

lats = df_geo['latitude_sirgas2000'].values
lons = df_geo['longitude_sirgas2000'].values

# ── KDE com scipy ────────────────────────────────────────────────────────────
# Repetir pontos de acordo com o peso (ponderação)
lats_w = np.repeat(lats, pesos.astype(int))
lons_w = np.repeat(lons, pesos.astype(int))

kde = gaussian_kde(np.vstack([lats_w, lons_w]), bw_method='scott')

# Grade de avaliação (100x100 pontos sobre o bounding box)
lat_min, lat_max = lats.min() - 0.005, lats.max() + 0.005
lon_min, lon_max = lons.min() - 0.005, lons.max() + 0.005
n_grid = 100
lat_grid = np.linspace(lat_min, lat_max, n_grid)
lon_grid = np.linspace(lon_min, lon_max, n_grid)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
Z = kde(np.vstack([LAT.ravel(), LON.ravel()])).reshape(n_grid, n_grid)

# Normalizar 0-1
Z_norm = (Z - Z.min()) / (Z.max() - Z.min())

# ── Mapa KDE com Folium ───────────────────────────────────────────────────────
mapa_kde = folium.Map(
    location=[centro_lat, centro_lon], zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri'
)

# Converter grade KDE em lista [lat, lon, peso] para HeatMap
heat_kde = []
threshold = 0.05  # só células com densidade > 5% do máximo
for i in range(n_grid):
    for j in range(n_grid):
        v = float(Z_norm[i, j])
        if v > threshold:
            heat_kde.append([float(lat_grid[i]), float(lon_grid[j]), v])

HeatMap(
    heat_kde,
    radius=20, blur=15, max_zoom=16, min_opacity=0.3,
    gradient={'0.2': 'blue', '0.5': 'yellow', '0.8': 'orange', '1.0': 'red'}
).add_to(mapa_kde)

# Marcar os 10 pontos de maior densidade
flat_idx = np.argsort(Z_norm.ravel())[::-1][:10]
for rank, idx in enumerate(flat_idx, 1):
    i, j = divmod(idx, n_grid)
    folium.CircleMarker(
        location=[lat_grid[i], lon_grid[j]],
        radius=8, color='darkred', fill=True, fill_color='darkred', fill_opacity=0.9,
        tooltip=f'Hotspot #{rank} — densidade {Z_norm[i,j]:.2f}'
    ).add_to(mapa_kde)

mapa_kde.save('outputs/mapa_kde.html')
print(f'Salvo: outputs/mapa_kde.html  |  {len(heat_kde)} células na grade KDE')

# ── Top ruas por densidade de acidentes ──────────────────────────────────────
top_ruas = (
    df_geo.groupby('logradouro_nome')
    .agg(total=('numero_ocorrencia_associado','count'),
         fatais=('qtd_fatais','sum'),
         feridos=('qtd_feridos','sum'))
    .sort_values('total', ascending=False)
    .head(15)
    .reset_index()
)
top_ruas['indice_risco'] = top_ruas['total'] + top_ruas['fatais']*4 + top_ruas['feridos']*2
top_ruas = top_ruas.sort_values('indice_risco', ascending=False)
print('\nTop 15 ruas por índice de risco (acidentes + ponderação por gravidade):')
print(top_ruas.to_string(index=False))
top_ruas.to_csv('outputs/top_ruas_risco.csv', sep=';', index=False, encoding='utf-8-sig')
mapa_kde


## 9. Análise Espacial — Getis-Ord Gi*

O **Getis-Ord Gi*** é um indicador de autocorrelação espacial local que identifica
**hotspots** (clusters de alta concentração) e **coldspots** (clusters de baixa concentração)
com significância estatística.

- Cada ponto de acidente é avaliado em relação aos seus vizinhos dentro de um raio
- O resultado é um **z-score**: quanto mais alto (> 1.96), maior a confiança de que é hotspot
- Níveis de confiança: 90% (|z|>1.65), 95% (|z|>1.96), 99% (|z|>2.58)

Diferença do KDE: o KDE mostra densidade visual; o Gi* **prova estatisticamente** que
um local é mais perigoso do que seria esperado ao acaso.


In [ ]:
import libpysal
from esda.getisord import G_Local
import numpy as np
import folium
import pandas as pd
import warnings

# Base: acidentes com coordenada
df_gi = df_geo[['latitude_sirgas2000','longitude_sirgas2000','severidade_maxima','logradouro_nome']].copy().reset_index(drop=True)

# Variável de interesse: peso de severidade
peso_map = {'FATAL': 5, 'GRAVE': 4, 'LEVE': 3, 'NAO INFORMADO': 2, 'SEM VITIMA': 1}
df_gi['peso'] = df_gi['severidade_maxima'].map(peso_map).fillna(1)

# ── Matriz de pesos espaciais ─────────────────────────────────────────────────
# Raio de 0.009 graus ≈ 1km — garante que acidentes em vias próximas se conectem
# e elimina ilhas (pontos isolados sem vizinhos)
coords = list(zip(df_gi['latitude_sirgas2000'], df_gi['longitude_sirgas2000']))
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    w = libpysal.weights.DistanceBand.from_array(
        np.array(coords), threshold=0.009, binary=True, silence_warnings=True
    )
    w.transform = 'r'

n_islands = len(w.islands)
print(f'Vizinhos por ponto — média: {np.mean([len(v) for v in w.neighbors.values()]):.1f} | ilhas (sem vizinho): {n_islands}')

# ── Getis-Ord Gi* ─────────────────────────────────────────────────────────────
# star=None usa os valores da diagonal já definidos pela matriz
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    g_local = G_Local(df_gi['peso'].values, w, transform='r', star=None, permutations=499)

df_gi['gi_zscore'] = g_local.Zs
df_gi['gi_pvalue'] = g_local.p_sim

# Classificar hotspot/coldspot por nível de confiança
def classif_gi(z, p):
    if pd.isna(z) or p > 0.10: return 'NAO_SIGNIFICATIVO'
    if z > 2.58:   return 'HOTSPOT_99'
    if z > 1.96:   return 'HOTSPOT_95'
    if z > 1.65:   return 'HOTSPOT_90'
    if z < -2.58:  return 'COLDSPOT_99'
    if z < -1.96:  return 'COLDSPOT_95'
    if z < -1.65:  return 'COLDSPOT_90'
    return 'NAO_SIGNIFICATIVO'

df_gi['cluster'] = [classif_gi(z, p) for z, p in zip(df_gi['gi_zscore'], df_gi['gi_pvalue'])]

print('\nDistribuição dos clusters Gi*:')
print(df_gi['cluster'].value_counts())

# ── Mapa Gi* ──────────────────────────────────────────────────────────────────
cor_gi = {
    'HOTSPOT_99':       '#d73027',
    'HOTSPOT_95':       '#fc8d59',
    'HOTSPOT_90':       '#fee090',
    'COLDSPOT_90':      '#e0f3f8',
    'COLDSPOT_95':      '#91bfdb',
    'COLDSPOT_99':      '#4575b4',
    'NAO_SIGNIFICATIVO':'#cccccc',
}

mapa_gi = folium.Map(
    location=[centro_lat, centro_lon], zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri'
)

for _, row in df_gi.iterrows():
    cluster = row['cluster']
    cor     = cor_gi.get(cluster, '#cccccc')
    radius  = 8 if 'HOTSPOT' in cluster else (6 if 'COLDSPOT' in cluster else 4)
    via     = str(row.get('logradouro_nome','')) if pd.notna(row.get('logradouro_nome')) else 'Nao informado'
    folium.CircleMarker(
        location=[row['latitude_sirgas2000'], row['longitude_sirgas2000']],
        radius=radius,
        color=cor, fill=True, fill_color=cor, fill_opacity=0.8,
        tooltip=f"{cluster} | z={row['gi_zscore']:.2f} | {via}"
    ).add_to(mapa_gi)

legenda_gi = """
<div style="position:fixed;bottom:40px;left:40px;background:white;
            padding:12px;border:1px solid #ccc;border-radius:6px;font-size:12px;z-index:999">
  <b>Getis-Ord Gi*</b><br>
  <span style='color:#d73027'>&#9679;</span> Hotspot 99%<br>
  <span style='color:#fc8d59'>&#9679;</span> Hotspot 95%<br>
  <span style='color:#fee090'>&#9679;</span> Hotspot 90%<br>
  <span style='color:#cccccc'>&#9679;</span> Nao significativo<br>
  <span style='color:#91bfdb'>&#9679;</span> Coldspot 95%<br>
  <span style='color:#4575b4'>&#9679;</span> Coldspot 99%
</div>
"""
mapa_gi.get_root().html.add_child(folium.Element(legenda_gi))
mapa_gi.save('outputs/mapa_gi_star.html')
print('\nSalvo: outputs/mapa_gi_star.html')

# ── Ruas com mais hotspots significativos ─────────────────────────────────────
hotspots = df_gi[df_gi['cluster'].str.contains('HOTSPOT')]
print(f'Total de pontos hotspot: {len(hotspots)}')
if len(hotspots) > 0:
    print('\nRuas com mais hotspots:')
    print(hotspots['logradouro_nome'].value_counts().head(10).to_string())

df_gi.to_csv('outputs/gi_star_resultados.csv', sep=';', index=False, encoding='utf-8-sig')
print('Salvo: outputs/gi_star_resultados.csv')
mapa_gi


## 10. Mineração de Regras de Associação — FP-Growth e Apriori

Cada acidente é tratado como uma **transação**: um conjunto de atributos categóricos.
Os algoritmos buscam padrões do tipo:

> Quando o acidente ocorre em AVENIDA + NOITE + MOTO, em 72% dos casos a severidade é LEVE

**Métricas usadas:**
| Métrica | Significado | Limiar |
|---|---|---|
| **Suporte** | % de acidentes onde o padrão ocorre | ≥ 5% |
| **Confiança** | % das vezes que Y ocorre dado X | ≥ 60% |
| **Lift** | Quanto Y é mais provável dado X vs. aleatório | > 1.0 |

**Colunas usadas na mineração** (sem nulos, presentes em todos os 1.012 acidentes):
`faixa_horaria`, `dia_semana_cat`, `severidade_cat`, `logradouro_cat`,
`motivo_cat`, `qtd_veiculos_cat`, `tipo_veiculo_cat`, `tem_pedestre`, `mes_cat`


In [ ]:
from mlxtend.frequent_patterns import fpgrowth, apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import os
os.makedirs('outputs', exist_ok=True)

# ── 10.1 Construção da base transacional ─────────────────────────────────────
# Colunas sem nulos presentes nos 1.012 acidentes
cols_transacao = [
    'faixa_horaria', 'dia_semana_cat', 'severidade_cat',
    'logradouro_cat', 'motivo_cat', 'qtd_veiculos_cat',
    'tipo_veiculo_cat', 'tem_pedestre', 'mes_cat'
]

# Prefixar cada valor com o nome da coluna para evitar colisões
# Ex: 'TARDE' → 'hora=TARDE', 'AVENIDA' → 'via=AVENIDA'
prefixos = {
    'faixa_horaria':    'hora',
    'dia_semana_cat':   'semana',
    'severidade_cat':   'sev',
    'logradouro_cat':   'via',
    'motivo_cat':       'motivo',
    'qtd_veiculos_cat': 'veic',
    'tipo_veiculo_cat': 'tipo',
    'tem_pedestre':     'pedestre',
    'mes_cat':          'trimestre',
}

transacoes = []
for _, row in df[cols_transacao].dropna().iterrows():
    items = [f"{prefixos[col]}={row[col]}" for col in cols_transacao]
    transacoes.append(items)

print(f'Total de transações: {len(transacoes)}')

# Codificar em matriz binária
te = TransactionEncoder()
te_array = te.fit_transform(transacoes)
df_transac = pd.DataFrame(te_array, columns=te.columns_)
print(f'Dimensão da matriz transacional: {df_transac.shape}')

# ── 10.2 FP-Growth ───────────────────────────────────────────────────────────
print('\n--- FP-Growth ---')
freq_fp = fpgrowth(df_transac, min_support=0.05, use_colnames=True)
freq_fp = freq_fp.sort_values('support', ascending=False)
print(f'Itemsets frequentes encontrados: {len(freq_fp)}')

regras_fp = association_rules(freq_fp, metric='confidence', min_threshold=0.60, num_itemsets=len(freq_fp))
regras_fp = regras_fp[regras_fp['lift'] > 1.0].sort_values('lift', ascending=False)
print(f'Regras válidas (confiança≥60%, lift>1): {len(regras_fp)}')

# ── 10.3 Apriori ─────────────────────────────────────────────────────────────
print('\n--- Apriori ---')
freq_ap = apriori(df_transac, min_support=0.05, use_colnames=True)
freq_ap = freq_ap.sort_values('support', ascending=False)
print(f'Itemsets frequentes encontrados: {len(freq_ap)}')

regras_ap = association_rules(freq_ap, metric='confidence', min_threshold=0.60, num_itemsets=len(freq_ap))
regras_ap = regras_ap[regras_ap['lift'] > 1.0].sort_values('lift', ascending=False)
print(f'Regras válidas (confiança≥60%, lift>1): {len(regras_ap)}')

# ── 10.4 Top regras ───────────────────────────────────────────────────────────
print('\n=== TOP 20 REGRAS POR LIFT (FP-Growth) ===')
cols_exib = ['antecedents','consequents','support','confidence','lift']
top20 = regras_fp.head(20)[cols_exib].copy()
top20['antecedents'] = top20['antecedents'].apply(lambda x: ', '.join(sorted(x)))
top20['consequents'] = top20['consequents'].apply(lambda x: ', '.join(sorted(x)))
top20['support']     = top20['support'].round(3)
top20['confidence']  = top20['confidence'].round(3)
top20['lift']        = top20['lift'].round(3)
print(top20.to_string(index=False))

# ── 10.5 Regras focadas em SEVERIDADE ────────────────────────────────────────
print('\n=== REGRAS CUJO CONSEQUENTE É SEVERIDADE ===')
regras_sev = regras_fp[
    regras_fp['consequents'].apply(lambda x: any('sev=' in i for i in x))
].sort_values('lift', ascending=False).head(20)

if len(regras_sev) > 0:
    r = regras_sev[cols_exib].copy()
    r['antecedents'] = r['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    r['consequents'] = r['consequents'].apply(lambda x: ', '.join(sorted(x)))
    r['support']     = r['support'].round(3)
    r['confidence']  = r['confidence'].round(3)
    r['lift']        = r['lift'].round(3)
    print(r.to_string(index=False))
else:
    print('Nenhuma regra com consequente de severidade nos limiares atuais.')
    print('Tente reduzir min_support para 0.03 ou min_confidence para 0.50.')

# ── 10.6 Exportar ─────────────────────────────────────────────────────────────
def exportar_regras(regras, nome):
    r = regras.copy()
    r['antecedents'] = r['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    r['consequents'] = r['consequents'].apply(lambda x: ', '.join(sorted(x)))
    r.to_csv(f'outputs/{nome}.csv', sep=';', index=False, encoding='utf-8-sig')
    print(f'Salvo: outputs/{nome}.csv  ({len(r)} regras)')

exportar_regras(regras_fp, 'regras_fpgrowth')
exportar_regras(regras_ap, 'regras_apriori')


## 11. Mapas — Visualização Espacial dos Acidentes

Usamos apenas os **977 registros com coordenadas válidas**.
Três mapas são gerados:

| Mapa | O que mostra |
|------|--------------|
| **Mapa 1 — Severidade** | Marcadores coloridos por severidade do acidente |
| **Mapa 2 — Heatmap** | Densidade de acidentes (zonas mais quentes = mais acidentes) |
| **Mapa 3 — Cluster** | Agrupamento interativo, clique para abrir detalhes |

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster
import os
os.makedirs("outputs", exist_ok=True)

# Base espacial: apenas registros com coordenadas
df_geo = df.dropna(subset=["latitude_sirgas2000", "longitude_sirgas2000"]).copy()

# Fix #8: usar mediana em vez de média — robusta a coordenadas outlier
centro_lat = df_geo["latitude_sirgas2000"].median()
centro_lon = df_geo["longitude_sirgas2000"].median()

print(f"Registros com coordenada: {len(df_geo)}")
print(f"Centro do mapa (mediana): ({centro_lat:.5f}, {centro_lon:.5f})")

### Mapa 1 — Acidentes por Severidade

Cada ponto = 1 acidente. Cor indica a severidade:
🔴 Fatal | 🟠 Grave | 🔵 Leve | ⚫ Sem vítima / Não informado

Clique em qualquer ponto para ver: via, bairro, data/hora e veículos.

In [12]:
cor_severidade = {
    "FATAL":         "red",
    "GRAVE":         "orange",
    "LEVE":          "blue",
    "NAO INFORMADO": "gray",
    "SEM VITIMA":    "lightgray",
}

mapa1 = folium.Map(location=[centro_lat, centro_lon], zoom_start=13,
                   tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                   attr='Tiles &copy; Esri')

for _, row in df_geo.iterrows():
    sev    = row["severidade_maxima"]
    cor    = cor_severidade.get(sev, "gray")
    via    = str(row.get("logradouro_nome", "")).title() if pd.notna(row.get("logradouro_nome")) else "Não informado"
    bairro = str(row.get("nome_bairro", "")).title()    if pd.notna(row.get("nome_bairro"))    else "Não informado"
    hora   = str(row.get("horario_fato", ""))[:5]
    data   = str(row.get("data_fato", ""))[:10]

    popup_txt = (
        f"<b>{sev}</b><br>"
        f"Via: {via}<br>"
        f"Bairro: {bairro}<br>"
        f"Data: {data} às {hora}<br>"
        f"Veículos: {row.get('qtd_veiculos', 0)}"
    )

    folium.CircleMarker(
        location=[row["latitude_sirgas2000"], row["longitude_sirgas2000"]],
        radius=5,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        popup=folium.Popup(popup_txt, max_width=220),
        tooltip=f"{sev} — {via}"
    ).add_to(mapa1)

legenda_html = """
<div style="position:fixed;bottom:40px;left:40px;background:white;
            padding:10px;border:1px solid #ccc;border-radius:6px;font-size:13px;z-index:999">
  <b>Severidade</b><br>
  <span style='color:red'>&#9679;</span> Fatal<br>
  <span style='color:orange'>&#9679;</span> Grave<br>
  <span style='color:blue'>&#9679;</span> Leve<br>
  <span style='color:gray'>&#9679;</span> N\u00e3o informado / Sem v\u00edtima
</div>
"""
mapa1.get_root().html.add_child(folium.Element(legenda_html))
mapa1.save("outputs/mapa_severidade.html")
print(f"Salvo: outputs/mapa_severidade.html  ({len(df_geo)} pontos)")
mapa1

Salvo: outputs/mapa_severidade.html  (977 pontos)


### Mapa 2 — Heatmap de Densidade

Mostra onde os acidentes se concentram espacialmente.
Zonas vermelhas/amarelas = maior densidade de ocorrências.
O peso de cada ponto é proporcional à severidade (fatais pesam mais).

In [13]:
mapa2 = folium.Map(location=[centro_lat, centro_lon], zoom_start=13,
                   tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}',
                   attr='Tiles &copy; Esri')

peso_sev = {"FATAL": 4, "GRAVE": 3, "LEVE": 2, "NAO INFORMADO": 1, "SEM VITIMA": 1}

heat_data = [
    [row["latitude_sirgas2000"],
     row["longitude_sirgas2000"],
     peso_sev.get(row["severidade_maxima"], 1)]
    for _, row in df_geo.iterrows()
]

HeatMap(
    heat_data,
    radius=18,
    blur=12,
    max_zoom=15,
    min_opacity=0.3,
    gradient={"0.4": "blue", "0.65": "yellow", "1.0": "red"}
).add_to(mapa2)

mapa2.save("outputs/mapa_heatmap.html")
print("Salvo: outputs/mapa_heatmap.html")
mapa2

Salvo: outputs/mapa_heatmap.html


### Mapa 3 — Cluster Interativo

Agrupa acidentes próximos. Aproxime o zoom para ver os pontos individuais.
Cada marcador tem popup completo com via, bairro, data, veículos e motivo.

In [14]:
mapa3 = folium.Map(location=[centro_lat, centro_lon], zoom_start=13,
                   tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                   attr='Tiles &copy; Esri')
cluster = MarkerCluster(name="Acidentes").add_to(mapa3)

icone_cor = {
    "FATAL":         "red",
    "GRAVE":         "orange",
    "LEVE":          "blue",
    "NAO INFORMADO": "gray",
    "SEM VITIMA":    "lightgray",
}

for _, row in df_geo.iterrows():
    sev    = row["severidade_maxima"]
    cor    = icone_cor.get(sev, "gray")
    via    = str(row.get("logradouro_nome", "")).title() if pd.notna(row.get("logradouro_nome")) else "N\u00e3o informado"
    bairro = str(row.get("nome_bairro", "")).title()    if pd.notna(row.get("nome_bairro"))    else "N\u00e3o informado"
    hora   = str(row.get("horario_fato", ""))[:5]
    data   = str(row.get("data_fato", ""))[:10]
    veic   = str(row.get("tipos_veiculos", "")) if pd.notna(row.get("tipos_veiculos")) else "\u2014"
    motivo = str(row.get("motivo_presumido_descricao", "")).title() if pd.notna(row.get("motivo_presumido_descricao")) else "\u2014"

    popup_txt = (
        f"<b>Severidade:</b> {sev}<br>"
        f"<b>Via:</b> {via}<br>"
        f"<b>Bairro:</b> {bairro}<br>"
        f"<b>Data/hora:</b> {data} \u00e0s {hora}<br>"
        f"<b>Ve\u00edculos:</b> {veic}<br>"
        f"<b>Motivo:</b> {motivo}"
    )

    folium.Marker(
        location=[row["latitude_sirgas2000"], row["longitude_sirgas2000"]],
        popup=folium.Popup(popup_txt, max_width=250),
        tooltip=f"{sev} \u2014 {via}",
        icon=folium.Icon(color=cor, icon="info-sign")
    ).add_to(cluster)

folium.LayerControl().add_to(mapa3)
mapa3.save("outputs/mapa_cluster.html")
print("Salvo: outputs/mapa_cluster.html")
mapa3

Salvo: outputs/mapa_cluster.html


## 12. Exportação dos datasets

In [15]:
import os
os.makedirs("outputs", exist_ok=True)

df.to_csv("outputs/joao_monlevade_2025_tratado.csv", sep=";", index=False, encoding="utf-8-sig")
df_geo.to_csv("outputs/joao_monlevade_2025_geo.csv", sep=";", index=False, encoding="utf-8-sig")

print(f"Dataset completo:  outputs/joao_monlevade_2025_tratado.csv  \u2192  {df.shape}")
print(f"Dataset espacial:  outputs/joao_monlevade_2025_geo.csv      \u2192  {df_geo.shape}")
print("\nMapas em outputs/:")
for arq in ["mapa_severidade.html", "mapa_heatmap.html", "mapa_cluster.html"]:
    if os.path.exists(f"outputs/{arq}"):
        kb = os.path.getsize(f"outputs/{arq}") / 1024
        print(f"  {arq}  ({kb:.0f} KB)")

Dataset completo:  outputs/joao_monlevade_2025_tratado.csv  →  (1012, 48)
Dataset espacial:  outputs/joao_monlevade_2025_geo.csv      →  (977, 48)

Mapas em outputs/:
  mapa_severidade.html  (1321 KB)
  mapa_heatmap.html  (47 KB)
  mapa_cluster.html  (1514 KB)
